In [4]:
import sys
!"{sys.executable}" -m pip install networkx numpy scipy torch scikit-learn matplotlib
# torch-geometric may require special install steps on Windows and is optional for this notebook



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [6]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load Budapest edgelist as an unweighted graph
path = os.path.join("..", "Datasets", "Budapest.txt")

# Each line: V1 V2 weight. We ignore the weight and use binary edges.
edges = np.loadtxt(path, dtype=int, usecols=(0, 1))

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")

Nodes: 480
Edges: 1000
Graph is large; skipping full plot.


In [7]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [8]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([14, 96,  7,  7,  6,  2, 14, 67,  5,  4,  1,  8,  4, 32,  6,  2,  9,
        2, 13,  2,  4, 89,  7, 62,  3,  2,  6,  1,  1,  6,  5,  9,  6, 10,
        2, 36,  5,  1,  7,  1,  5,  2,  2,  4,  4,  1,  3,  7,  1,  1,  2,
        1, 10,  5,  1,  3,  4,  1,  3,  1,  3,  2,  4,  7,  7, 30,  2,  1,
        5,  9, 14,  3, 18,  3,  2,  2,  5,  3,  6,  2,  2,  1,  4, 17,  2,
        2,  9,  1,  5,  1,  5,  4,  6,  3, 16,  1,  6,  6,  8,  2,  2,  8,
        3,  5,  3,  7,  9,  6,  1,  2,  2,  1,  9,  9,  4,  2,  3,  2,  5,
        2,  1,  3,  1,  6,  2,  4,  4,  9,  5,  9,  3,  3,  2, 12,  5,  9,
        3,  2,  8,  1,  1,  2,  3,  4,  1,  2,  8,  2,  1,  4, 13, 13,  4,
        3,  2,  4,  8,  2,  6,  6,  1, 11,  8,  1,  3,  2,  4,  6,  8,  7,
        1,  2,  4,  4,  1,  2,  3,  2,  4,  5,  3,  1,  9,  5,  2,  1,  2,
        3,  5,  1,  1,  4,  2,  3,  7,  2,  2,  1,  4,  1,  4,  1,  2,  7,
        5,  4,  1,  4,  1,  1,  3,  8,  1,  1,  1,  5,  4,  3,  4,  8,  2,
        1, 11,  3,  7,  5

In [9]:
dist = dict(nx.all_pairs_shortest_path_length(G))
dist

{np.int64(127): {np.int64(127): 0,
  np.int64(504): 1,
  np.int64(145): 1,
  np.int64(131): 1,
  np.int64(494): 1,
  np.int64(189): 1,
  np.int64(505): 1,
  np.int64(503): 1,
  np.int64(146): 1,
  np.int64(493): 1,
  np.int64(124): 1,
  np.int64(107): 1,
  np.int64(112): 1,
  np.int64(121): 1,
  np.int64(111): 1,
  np.int64(29): 2,
  np.int64(226): 2,
  np.int64(58): 2,
  np.int64(153): 2,
  np.int64(15): 2,
  np.int64(76): 2,
  np.int64(129): 2,
  np.int64(495): 2,
  np.int64(218): 2,
  np.int64(14): 2,
  np.int64(49): 2,
  np.int64(486): 2,
  np.int64(8): 2,
  np.int64(193): 2,
  np.int64(180): 2,
  np.int64(488): 2,
  np.int64(77): 2,
  np.int64(12): 2,
  np.int64(237): 2,
  np.int64(144): 2,
  np.int64(254): 2,
  np.int64(496): 2,
  np.int64(235): 2,
  np.int64(213): 2,
  np.int64(94): 2,
  np.int64(51): 2,
  np.int64(151): 2,
  np.int64(78): 2,
  np.int64(60): 2,
  np.int64(82): 2,
  np.int64(126): 2,
  np.int64(7): 2,
  np.int64(500): 2,
  np.int64(255): 2,
  np.int64(182): 2,
  

In [10]:
n = len(nodelist)
dist_matrix = np.zeros((n, n))

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and v in dist[u]:
            dist_matrix[i, j] = dist[u][v]

print(dist_matrix)

[[0. 1. 1. ... 4. 4. 2.]
 [1. 0. 1. ... 3. 4. 2.]
 [1. 1. 0. ... 3. 5. 3.]
 ...
 [4. 3. 3. ... 0. 6. 4.]
 [4. 4. 5. ... 6. 0. 4.]
 [2. 2. 3. ... 4. 4. 0.]]


4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [11]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [325.15527987 423.7776955  283.64157716 318.90829392 279.84043924
 291.95531089 338.13975019 444.94266699 282.23754167 278.62032979
 209.32570023 285.38236064 252.48528915 358.29700802 287.34750564
 167.23372689 208.37852629 233.37781924 325.94082288 228.08559768
 281.93403206 429.91691021 315.01750102 438.09518663 167.05555304
 165.22321121 204.39436613 276.27391336 288.34026632 283.96836301
 281.88882173 319.15914087 314.75564251 288.06697461 251.75664271
 360.9950523  312.19263986 225.9164207  315.52933177 288.34026632
 315.21469536 310.06743544 210.04014443 218.05012418 280.93583683
 273.42560043 275.8411713  282.62033014 273.42560043 273.42560043
 274.301496   212.46932288 291.23808385 314.2853529  248.97560356
 209.01853213 279.68691027 167.11892207 214.80793694 273.42560043
 250.71788595 249.73282093 281.78117493 321.38026845 316.41498529
 401.95440259 293.12335655 273.42560043 315.95561576 285.5090027
 327.06157356 275.74661979 336.10327105 275.74661979 274.61670984
 274.5

### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [12]:
NLI = np.zeros(n)

for i, u in enumerate(nodelist):

    neighbors = list(G.neighbors(u))

    # Sum of exponential terms (using neighbor count here as influence proxy)
    exp_sum = 0
    for v in neighbors:
        exp_sum += np.exp(G.degree(v))   # you can modify this part if Ne_i(v_i) defined differently

    if exp_sum > 0:
        NLI[i] = (G.degree(u) * np.log2(exp_sum)) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

#calculation Verified

NLI: [4.03954611e+00 2.76997448e+01 2.01977306e+00 2.01977306e+00
 1.73123405e+00 4.02752366e-01 4.03954611e+00 1.93321135e+01
 1.44269504e+00 1.15415603e+00 2.40449173e-02 2.30831207e+00
 3.84718678e-01 8.55999058e+00 5.79366998e-01 5.41065431e-02
 1.55174671e-01 7.81460818e-02 3.47749617e+00 3.61764794e-02
 1.06999882e+00 2.38074738e+01 1.87249794e+00 1.65849817e+01
 5.46951289e-02 3.63594457e-02 8.70668347e-02 2.67499705e-01
 2.01376183e-01 1.60499823e+00 1.33749853e+00 2.40749735e+00
 1.60499823e+00 2.67499705e+00 2.16404256e-01 1.03874043e+01
 1.33749853e+00 1.50280733e-02 1.87249794e+00 2.01376183e-01
 1.33749853e+00 5.77078016e-01 2.82115840e-02 5.06265449e-02
 1.06999882e+00 2.88539008e-01 8.65617025e-01 2.01977306e+00
 2.88539008e-01 2.88539008e-01 5.77078016e-01 3.00561467e-02
 2.67499705e+00 1.44269504e+00 9.61796694e-02 3.92143647e-02
 1.06999882e+00 9.01684401e-03 9.91965242e-02 2.88539008e-01
 2.88539008e-01 1.92359339e-01 1.06999882e+00 1.87249794e+00
 1.87249794e+00 8.0

4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [13]:
NLGC = NLI * NGI
NLGC

array([1.31347975e+03, 1.17385340e+04, 5.72891615e+02, 6.44122380e+02,
       4.84469297e+02, 1.17585692e+02, 1.36593111e+03, 8.60168216e+03,
       4.07182702e+02, 3.21571334e+02, 5.03321916e+00, 6.58751546e+02,
       9.71358065e+01, 3.06701901e+03, 1.66479662e+02, 9.04843885e+00,
       3.23350693e+01, 1.82375621e+01, 1.13345796e+03, 8.25133393e+00,
       3.01669082e+02, 1.02352356e+04, 5.89869621e+02, 7.26580067e+03,
       9.13712501e+00, 6.00742438e+00, 1.77959705e+01, 7.39031905e+01,
       5.80648622e+01, 4.55768721e+02, 3.77025884e+02, 7.68374786e+02,
       5.05182250e+02, 7.70578309e+02, 5.44812090e+01, 3.74980156e+03,
       4.17557196e+02, 3.39508854e+00, 5.90828023e+02, 5.80648622e+01,
       4.21599191e+02, 1.78933101e+02, 5.92556518e+00, 1.10391244e+01,
       3.00601014e+02, 7.88939516e+01, 2.38772814e+02, 5.70828928e+02,
       7.88939516e+01, 7.88939516e+01, 1.58293363e+02, 6.38600913e+00,
       7.79061017e+02, 4.53417920e+02, 2.39463912e+01, 8.19652896e+00,
      

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [14]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [4.03954611e+00 2.76997448e+01 2.01977306e+00 2.01977306e+00
 1.73123405e+00 4.02752366e-01 4.03954611e+00 1.93321135e+01
 1.44269504e+00 1.15415603e+00 2.40449173e-02 2.30831207e+00
 3.84718678e-01 8.55999058e+00 5.79366998e-01 5.41065431e-02
 1.55174671e-01 7.81460818e-02 3.47749617e+00 3.61764794e-02
 1.06999882e+00 2.38074738e+01 1.87249794e+00 1.65849817e+01
 5.46951289e-02 3.63594457e-02 8.70668347e-02 2.67499705e-01
 2.01376183e-01 1.60499823e+00 1.33749853e+00 2.40749735e+00
 1.60499823e+00 2.67499705e+00 2.16404256e-01 1.03874043e+01
 1.33749853e+00 1.50280733e-02 1.87249794e+00 2.01376183e-01
 1.33749853e+00 5.77078016e-01 2.82115840e-02 5.06265449e-02
 1.06999882e+00 2.88539008e-01 8.65617025e-01 2.01977306e+00
 2.88539008e-01 2.88539008e-01 5.77078016e-01 3.00561467e-02
 2.67499705e+00 1.44269504e+00 9.61796694e-02 3.92143647e-02
 1.06999882e+00 9.01684401e-03 9.91965242e-02 2.88539008e-01
 2.88539008e-01 1.92359339e-01 1.06999882e+00 1.87249794e+00
 1.87249794e+00 

### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [15]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Nodes used: [np.int64(127), np.int64(504), np.int64(503), np.int64(505), np.int64(131), np.int64(111), np.int64(124), np.int64(121), np.int64(145), np.int64(494), np.int64(146), np.int64(493), np.int64(107), np.int64(112), np.int64(189), None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [16]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [17]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[ 0.28853901 27.69974479  0.         ...  0.          0.
   0.        ]
 [ 0.28853901 27.69974479  0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]
E_NLI2:
 [[ 27.98828379 208.61370291   0.         ...   0.           0.
    0.        ]
 [ 27.98828379 208.61370291   0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 ...
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]]
E_NLI3:
 [[ 236.60198671 4842.81518272    0.         ...  

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [18]:
channels = []

for node in G.nodes():

    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1)
    tensor = np.stack([c1, c2, c3, c4, c5, c6])

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1)
channels = np.array(channels)

print(channels.shape)

(480, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [19]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [20]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [22]:
import numpy as np
import random

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)

beta = 1.5 * beta_c
mu = 1


# ---- SIR Simulation ----
def SIR_simulation(G, seed, beta, mu, steps=1000):

    susceptible = set(G.nodes())
    infected = {seed}
    recovered = set()

    susceptible.remove(seed)

    for _ in range(steps):

        new_infected = set()
        new_recovered = set()

        for node in infected:

            # spread infection
            for nbr in G.neighbors(node):
                if nbr in susceptible:
                    if random.random() < beta:
                        new_infected.add(nbr)

            # recovery
            if random.random() < mu:
                new_recovered.add(node)

        infected |= new_infected
        infected -= new_recovered

        recovered |= new_recovered
        susceptible -= new_infected

        if len(infected) == 0:
            break

    return len(recovered)


# ---- Label Generation ----
labels = []
runs = 500

for node in G.nodes():

    spread = 0

    for _ in range(runs):
        spread += SIR_simulation(G, node, beta, mu)

    labels.append(spread / runs)

labels = np.array(labels)


# ---- Normalize Labels ----
labels = labels / np.max(labels)

print("Labels:", labels)

Labels: [0.36634357 0.96006073 0.28179376 0.29685858 0.24687609 0.16115847
 0.46350578 0.96309705 0.21721359 0.22468761 0.07730935 0.2051851
 0.12483943 0.46806026 0.20378372 0.07474016 0.10708864 0.0902721
 0.39063412 0.08641831 0.18696718 1.         0.28459652 0.88134999
 0.07999533 0.073222   0.09938106 0.13102885 0.1698003  0.23683289
 0.20845498 0.33317762 0.29954455 0.27758963 0.11900035 0.49632138
 0.24454046 0.08653509 0.32675464 0.15158239 0.2986103  0.19374051
 0.07917786 0.08303165 0.17785823 0.14177274 0.17120168 0.26953171
 0.13219666 0.12191989 0.15134883 0.07625832 0.22281911 0.26357585
 0.08408268 0.08057924 0.17669041 0.06493052 0.10895714 0.11538012
 0.12285414 0.0947098  0.19771108 0.35828565 0.34859278 0.67966834
 0.18755109 0.14399159 0.30141306 0.24477403 0.38713068 0.1912881
 0.46432325 0.16582973 0.14994745 0.15648721 0.25528436 0.17750788
 0.28774962 0.20121453 0.07065281 0.0637627  0.30269765 0.5284363
 0.14259021 0.13651758 0.40616606 0.06644867 0.08945463 0.

In [23]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([-2.4518e-08,  3.4796e-09, -7.0443e-09, -4.4743e-08, -8.4078e-08,
         1.5734e-08])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.1600818932056427 y std: 0.12193291634321213
Epoch 0, Loss = 1.176339
Epoch 20, Loss = 0.110484
Epoch 40, Loss = 0.083711
Epoch 60, Loss = 0.074026
Epoch 80, Loss = 0.066552
Epoch 100, Loss = 0.060491
Epoch 120, Loss = 0.055162
Epoch 140, Loss = 0.050339
Epoch 160, Loss = 0.045956
Epoch 180, Loss = 0.041952
Epoch 200, Loss = 0.038266
Epoch 220, Loss = 0.034897
Epoch 240, Loss = 0.031809
Epoch 260, Loss = 0.028902
Epoch 280, Loss = 0.026200

Final Predictions:
 tensor([[ 1.6696e+00],
        [ 6.5377e+00],
        [ 7.4526e-01],
        [ 1.1947e+00],
        [ 6.1105e-01],
        [-3.8315e-02],
        [ 2.4084e+00],
        [ 6.5559e+00],
        [ 5.1985e-01],
        [ 3.6662e-01],
        [-5.0251e-01],
        [ 5.1913e-01],
        [-1.8181e-01],
        [ 2.5144e+00],
        [ 4.0

### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [24]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [ 1.66962421e+00  6.53765821e+00  7.45257735e-01  1.19465172e+00
  6.11046314e-01 -3.83154154e-02  2.40836906e+00  6.55585957e+00
  5.19852281e-01  3.66622984e-01 -5.02507567e-01  5.19127250e-01
 -1.81813449e-01  2.51435614e+00  4.03375745e-01 -5.02507567e-01
 -4.60330695e-01 -4.75937277e-01  1.83576834e+00 -5.00364959e-01
  3.50285470e-01  6.85206366e+00  1.06438196e+00  5.88330889e+00
 -5.02507567e-01 -5.02507567e-01 -4.93885487e-01 -2.06593007e-01
 -2.26246238e-01  7.88815975e-01  4.03120637e-01  1.38186944e+00
  1.20591009e+00  9.23864484e-01 -3.91531855e-01  2.74074602e+00
  7.50305057e-01 -5.02507567e-01  1.43141830e+00 -2.26246238e-01
  9.31305170e-01  3.84771943e-01 -5.02507567e-01 -5.02507567e-01
  1.15358472e-01 -1.69018298e-01  1.25719905e-01  9.23390865e-01
 -1.69018298e-01 -1.69018298e-01 -4.35694456e-02 -5.02507567e-01
  5.31293392e-01  8.77825975e-01 -4.84687746e-01 -5.01115799e-01
  1.97617590e-01 -5.02507567e-01 -4.29024667e-01 -1.69018298e-01
 -3.259

###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [25]:
from scipy.stats import kendalltau

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

spread_total = 0

for node in top_nodes:
    spread_total += SIR_simulation(G, node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)

Kendall Tau: 0.8785266380689339
Top-N node indices: [21  7  1]
Top-N nodes: [np.int64(1010), np.int64(503), np.int64(504)]
Spread ability: 71


In [26]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for budapest...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for budapest...
Kendall Tau (Prediction vs Degree): 0.5421
Kendall Tau (Prediction vs Betweenness): 0.2592
Kendall Tau (Prediction vs Closeness): 0.7967
Kendall Tau (Prediction vs PageRank): 0.2932
Kendall Tau (Prediction vs Coreness): 0.6228
Kendall Tau (Prediction vs Eigenvector): 0.8548


In [28]:
# ============================================================
# Weighted Centrality Correlation Analysis
# Weight Semantics: HIGH weight = ENEMIES (adversarial/costly)
# So weight is inversely proportional to influence strength.
# We convert: effective_weight = 1 / raw_weight
# This means a high-weight (enemy) edge contributes LESS
# to centrality — reflecting reduced influence flow.
# ============================================================

import numpy as np
import networkx as nx
from scipy.stats import kendalltau

# ---- Step 1: Reload the graph WITH weights ----
# Budapest.txt format: V1  V2  weight (tab/space separated)
path = os.path.join("..", "Datasets", "Budapest.txt")
raw = np.loadtxt(path)

if raw.ndim == 1:
    raw = raw.reshape(1, -1)

# Build a weighted graph
G_weighted = nx.Graph()
for row in raw:
    u, v, w = int(row[0]), int(row[1]), float(row[2])

    # Safety: avoid zero or negative weights before inversion
    w = abs(w) if w != 0 else 1e-6

    # Inverse weight: high raw weight → low influence (enemy logic)
    inv_w = 1.0 / w

    # If edge already exists keep the minimum inv_w (strongest enemy = weakest link)
    if G_weighted.has_edge(u, v):
        existing = G_weighted[u][v]['weight']
        G_weighted[u][v]['weight'] = min(existing, inv_w)
    else:
        G_weighted.add_edge(u, v, weight=inv_w)

# Remove self-loops (same as unweighted pipeline)
G_weighted.remove_edges_from(nx.selfloop_edges(G_weighted))

print(f"Weighted graph — Nodes: {G_weighted.number_of_nodes()}, Edges: {G_weighted.number_of_edges()}")
print(f"Sample inverse weights: {[round(G_weighted[u][v]['weight'], 4) for u,v in list(G_weighted.edges())[:5]]}")

# ---- Step 2: Compute Weighted Centrality Measures ----
print("\nCalculating weighted centrality measures for Budapest...")

# -- Weighted Degree (Strength) --
# Sum of inv_weights on edges — low-weight enemies reduce strength
strength_dict = dict(G_weighted.degree(weight='weight'))
strength_cent  = np.array([strength_dict.get(n, 0.0) for n in nodelist])
# Normalize to [0,1] for fair comparison
strength_cent  = strength_cent / strength_cent.max() if strength_cent.max() > 0 else strength_cent
tau_wdeg, _ = kendalltau(pred, strength_cent)
print(f"Kendall Tau (Prediction vs Weighted Degree / Strength): {tau_wdeg:.4f}")

# -- Weighted Betweenness --
# Uses inv_weight as distance — high-weight (enemy) edges are LONGER paths
# so they are avoided by shortest paths, reducing betweenness of bridge nodes
wbet_dict  = nx.betweenness_centrality(G_weighted, weight='weight', normalized=True)
wbet_cent  = np.array([wbet_dict[n] for n in nodelist])
tau_wbet, _ = kendalltau(pred, wbet_cent)
print(f"Kendall Tau (Prediction vs Weighted Betweenness):       {tau_wbet:.4f}")

# -- Weighted Closeness --
# distance = inv_weight → enemy edges make nodes "farther apart"
wclos_dict = nx.closeness_centrality(G_weighted, distance='weight')
wclos_cent = np.array([wclos_dict[n] for n in nodelist])
tau_wclos, _ = kendalltau(pred, wclos_cent)
print(f"Kendall Tau (Prediction vs Weighted Closeness):         {tau_wclos:.4f}")

# -- Weighted PageRank --
# edge weight = transition probability proxy (inv_w = low for enemy edges)
# so enemy edges pass less rank to neighbors
wpr_dict   = nx.pagerank(G_weighted, weight='weight')
wpr_cent   = np.array([wpr_dict[n] for n in nodelist])
tau_wpr, _ = kendalltau(pred, wpr_cent)
print(f"Kendall Tau (Prediction vs Weighted PageRank):          {tau_wpr:.4f}")

# -- Weighted Eigenvector Centrality --
# Propagates score proportional to inv_weight of connecting edges
# Enemy edges (high raw w → low inv_w) reduce neighbor's contribution
try:
    weig_dict  = nx.eigenvector_centrality(G_weighted, weight='weight', max_iter=1000)
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")
except nx.PowerIterationFailedConvergence:
    print("Weighted Eigenvector did not converge — trying numpy fallback...")
    weig_dict  = nx.eigenvector_centrality_numpy(G_weighted, weight='weight')
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")

# ---- Step 3: Side-by-side comparison table ----
print("\n" + "="*62)
print(f"{'Measure (Budapest)':<30} {'Unweighted':>12} {'Weighted':>12}")
print("="*62)
print(f"{'Degree / Strength':<30} {tau_deg:>12.4f} {tau_wdeg:>12.4f}")
print(f"{'Betweenness':<30} {tau_bet:>12.4f} {tau_wbet:>12.4f}")
print(f"{'Closeness':<30} {tau_clos:>12.4f} {tau_wclos:>12.4f}")
print(f"{'PageRank':<30} {tau_pr:>12.4f} {tau_wpr:>12.4f}")
print(f"{'Eigenvector':<30} {tau_eig:>12.4f} {tau_weig:>12.4f}")
print("="*62)
print("Weight semantics: high raw weight = adversarial edge")
print("Effective weight = 1 / raw_weight (enemy edges penalized)")

Weighted graph — Nodes: 480, Edges: 989
Sample inverse weights: [0.04, 0.2, 0.125, 0.1667, 0.3333]

Calculating weighted centrality measures for Budapest...
Kendall Tau (Prediction vs Weighted Degree / Strength): 0.3681
Kendall Tau (Prediction vs Weighted Betweenness):       0.0970
Kendall Tau (Prediction vs Weighted Closeness):         0.6985
Kendall Tau (Prediction vs Weighted PageRank):          0.2467
Kendall Tau (Prediction vs Weighted Eigenvector):       0.7683

Measure (Budapest)               Unweighted     Weighted
Degree / Strength                    0.5421       0.3681
Betweenness                          0.2592       0.0970
Closeness                            0.7967       0.6985
PageRank                             0.2932       0.2467
Eigenvector                          0.8548       0.7683
Weight semantics: high raw weight = adversarial edge
Effective weight = 1 / raw_weight (enemy edges penalized)
